In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except:
    ! pip install seaborn
    import seaborn as sns

#### Functions

#### Constants

In [ ]:
str_bucket = os.getcwd().split('/')[4].replace('_','-')
print(f'Bucket: {str_bucket}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_output = './output'

#### Output

In [ ]:
os.makedirs(str_output, exist_ok=True)

#### Import data

In [ ]:
str_filename = 'pitches'
str_uri = f's3://{str_bucket}/00_data_collection/{str_filename}'
df = pd.read_csv(str_uri)
print(f'Shape: {df.shape}')
df.head()

#### Data Overview

In [ ]:
# data types and non-null counts
df.info(verbose=True, show_counts=True)

In [ ]:
# missing values as a percentage
sr_missing = df.isnull().mean().sort_values(ascending=False)
sr_missing = sr_missing[sr_missing > 0]
print(f'{len(sr_missing)} columns with missing values:\n')
print(sr_missing.to_string())

In [ ]:
# numeric summary
df.describe()

#### Target Variable: pitch_type

In [ ]:
# pitch type mapping
dict_pitch_type = {
    'FF': 'Four-seam Fastball',
    'FT': 'Two-seam Fastball',
    'SI': 'Sinker',
    'FC': 'Cutter',
    'SL': 'Slider',
    'CH': 'Changeup',
    'CU': 'Curveball',
    'KC': 'Knuckle Curve',
    'FS': 'Splitter',
    'KN': 'Knuckleball',
    'FA': 'Fastball (generic)',
    'EP': 'Eephus',
    'SC': 'Screwball',
    'FO': 'Forkball',
    'IN': 'Intentional Ball',
    'PO': 'Pitchout',
    'AB': 'At-bat related',
    'UN': 'Unknown',
}

# distribution
sr_pitch_counts = df['pitch_type'].value_counts()
sr_pitch_pct = df['pitch_type'].value_counts(normalize=True) * 100

df_target = pd.DataFrame({
    'description': sr_pitch_counts.index.map(lambda x: dict_pitch_type.get(x, x)),
    'count': sr_pitch_counts.values,
    'pct': sr_pitch_pct.values.round(2),
})
df_target.index = sr_pitch_counts.index
df_target.index.name = 'pitch_type'
print(f'Missing pitch_type: {df["pitch_type"].isna().sum()} ({df["pitch_type"].isna().mean()*100:.2f}%)\n')
df_target

In [ ]:
# plot pitch type distribution
fig, ax = plt.subplots(figsize=(10, 5))
sr_pitch_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Pitch Type Distribution')
ax.set_xlabel('Pitch Type')
ax.set_ylabel('Count')
for i, (val, pct) in enumerate(zip(sr_pitch_counts.values, sr_pitch_pct.values)):
    ax.text(i, val + 3000, f'{pct:.1f}%', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{str_output}/pitch_type_distribution.png', dpi=150)
plt.show()

#### Available Features (available_prior_to_pitch = Yes)

These are the only columns we can use as model features without data leakage.

In [ ]:
# load metadata to identify available features
str_meta_uri = f's3://{str_bucket}/00_data_collection/pitch_by_pitch_metadata.csv'
df_meta = pd.read_csv(str_meta_uri, header=None, names=['column_name', 'available_prior', 'description'])

# columns available prior to pitch
list_available = df_meta[df_meta['available_prior'] == 'Yes']['column_name'].tolist()
list_not_available = df_meta[df_meta['available_prior'] == 'No']['column_name'].tolist()

print(f'Available prior to pitch ({len(list_available)} cols):')
print(list_available)
print(f'\nNOT available prior to pitch ({len(list_not_available)} cols):')
print(list_not_available)

#### Pitch Type by Count (Balls-Strikes)

In [ ]:
# filter to main pitch types for cleaner analysis
list_main_types = ['FF', 'SL', 'SI', 'FT', 'CH', 'CU', 'FC', 'FS']
df_main = df[df['pitch_type'].isin(list_main_types)].copy()
print(f'Rows with main pitch types: {len(df_main)} ({len(df_main)/len(df)*100:.1f}%)')

# create count string
df_main['count'] = df_main['balls'].astype(str) + '-' + df_main['strikes'].astype(str)

# pitch type distribution by count
df_count_pitch = pd.crosstab(df_main['count'], df_main['pitch_type'], normalize='index') * 100

# order counts logically
list_count_order = ['0-0', '0-1', '0-2', '1-0', '1-1', '1-2', '2-0', '2-1', '2-2', '3-0', '3-1', '3-2']
df_count_pitch = df_count_pitch.reindex(list_count_order)

fig, ax = plt.subplots(figsize=(12, 6))
df_count_pitch[list_main_types].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
ax.set_title('Pitch Type Distribution by Count')
ax.set_xlabel('Count (Balls-Strikes)')
ax.set_ylabel('Percentage')
ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{str_output}/pitch_type_by_count.png', dpi=150)
plt.show()

# also show the raw numbers
print('\nPitch type % by count:')
df_count_pitch[list_main_types].round(1)

#### Pitch Type by Handedness Matchup

In [ ]:
# pitcher hand vs batter hand matchup
df_main['matchup'] = df_main['p_throws'] + ' vs ' + df_main['stand']

df_matchup = pd.crosstab(df_main['matchup'], df_main['pitch_type'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(10, 5))
df_matchup[list_main_types].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
ax.set_title('Pitch Type Distribution by Handedness Matchup (Pitcher vs Batter)')
ax.set_xlabel('Matchup')
ax.set_ylabel('Percentage')
ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{str_output}/pitch_type_by_matchup.png', dpi=150)
plt.show()

print('\nPitch type % by matchup:')
df_matchup[list_main_types].round(1)

#### Pitch Type by Outs

In [ ]:
# pitch type by outs
df_outs = pd.crosstab(df_main['outs'], df_main['pitch_type'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(8, 5))
df_outs[list_main_types].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
ax.set_title('Pitch Type Distribution by Outs')
ax.set_xlabel('Outs')
ax.set_ylabel('Percentage')
ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{str_output}/pitch_type_by_outs.png', dpi=150)
plt.show()

print('\nPitch type % by outs:')
df_outs[list_main_types].round(1)

#### Pitcher Repertoire Diversity

In [ ]:
# how many distinct pitch types does each pitcher throw?
df_pitcher_repertoire = (
    df_main.groupby('pitcher_id')['pitch_type']
    .nunique()
    .reset_index()
    .rename(columns={'pitch_type': 'n_pitch_types'})
)

fig, ax = plt.subplots(figsize=(8, 4))
df_pitcher_repertoire['n_pitch_types'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Number of Distinct Pitch Types per Pitcher')
ax.set_xlabel('Number of Pitch Types')
ax.set_ylabel('Number of Pitchers')
plt.tight_layout()
plt.savefig(f'{str_output}/pitcher_repertoire_diversity.png', dpi=150)
plt.show()

print(f'Total pitchers: {df_pitcher_repertoire.shape[0]}')
print(f'\nDistribution:')
print(df_pitcher_repertoire['n_pitch_types'].value_counts().sort_index())

#### Top Pitcher Pitch Mix (Top 20 by Volume)

In [ ]:
# top 20 pitchers by pitch count - show their pitch mix
list_top_pitchers = df_main.groupby('pitcher_id').size().nlargest(20).index.tolist()
df_top = df_main[df_main['pitcher_id'].isin(list_top_pitchers)]

df_pitcher_mix = pd.crosstab(df_top['pitcher_id'], df_top['pitch_type'], normalize='index') * 100

# sort by fastball %
if 'FF' in df_pitcher_mix.columns:
    df_pitcher_mix = df_pitcher_mix.sort_values('FF', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
cols_present = [c for c in list_main_types if c in df_pitcher_mix.columns]
df_pitcher_mix[cols_present].plot(kind='barh', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
ax.set_title('Pitch Mix for Top 20 Pitchers (by volume)')
ax.set_xlabel('Percentage')
ax.set_ylabel('Pitcher ID')
ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{str_output}/top_pitcher_pitch_mix.png', dpi=150)
plt.show()

#### Pitch Type by Inning

In [ ]:
# pitch type by inning (cap at 9 for cleanliness)
df_inning = df_main[df_main['inning'] <= 9].copy()
df_inning_pitch = pd.crosstab(df_inning['inning'], df_inning['pitch_type'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(10, 5))
df_inning_pitch[list_main_types].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
ax.set_title('Pitch Type Distribution by Inning (1-9)')
ax.set_xlabel('Inning')
ax.set_ylabel('Percentage')
ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{str_output}/pitch_type_by_inning.png', dpi=150)
plt.show()

#### Pitch Type by Runners on Base

In [ ]:
# runners on base
df_main['runners_on'] = (
    df_main['on_1b'].notna().astype(int) +
    df_main['on_2b'].notna().astype(int) +
    df_main['on_3b'].notna().astype(int)
)

df_runners = pd.crosstab(df_main['runners_on'], df_main['pitch_type'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(8, 5))
df_runners[list_main_types].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
ax.set_title('Pitch Type Distribution by Runners on Base')
ax.set_xlabel('Number of Runners on Base')
ax.set_ylabel('Percentage')
ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{str_output}/pitch_type_by_runners.png', dpi=150)
plt.show()

print('\nPitch type % by runners on base:')
df_runners[list_main_types].round(1)

#### Pitcher Fatigue: Pitch Type by Pitch Count

In [ ]:
# bucket pitcher pitch count into groups
bins = [0, 25, 50, 75, 100, 125, 200]
labels = ['1-25', '26-50', '51-75', '76-100', '101-125', '126+']
df_main['pcount_bucket'] = pd.cut(df_main['pcount_pitcher'], bins=bins, labels=labels, right=True)

df_fatigue = pd.crosstab(df_main['pcount_bucket'], df_main['pitch_type'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(10, 5))
df_fatigue[list_main_types].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
ax.set_title('Pitch Type Distribution by Pitcher Pitch Count (game)')
ax.set_xlabel('Pitch Count Bucket')
ax.set_ylabel('Percentage')
ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{str_output}/pitch_type_by_pitcher_count.png', dpi=150)
plt.show()

print('\nPitch type % by pitcher pitch count bucket:')
df_fatigue[list_main_types].round(1)

#### First Pitch of At-Bat vs. Later Pitches

In [ ]:
# first pitch vs later pitches
df_main['is_first_pitch'] = (df_main['pcount_at_bat'] == 1).map({True: 'First Pitch', False: 'Later Pitch'})

df_first = pd.crosstab(df_main['is_first_pitch'], df_main['pitch_type'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(8, 4))
df_first[list_main_types].plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='black', linewidth=0.3)
ax.set_title('Pitch Type: First Pitch of At-Bat vs. Later Pitches')
ax.set_xlabel('')
ax.set_ylabel('Percentage')
ax.legend(title='Pitch Type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{str_output}/pitch_type_first_vs_later.png', dpi=150)
plt.show()

print('\nPitch type % first vs later:')
df_first[list_main_types].round(1)

#### Baseline Accuracy: Pitcher's Most Common Pitch Type

In [ ]:
# baseline 1: always predict the most common pitch type overall
str_most_common = df_main['pitch_type'].mode()[0]
flt_naive_acc = (df_main['pitch_type'] == str_most_common).mean() * 100
print(f'Baseline 1 - Always predict "{str_most_common}": {flt_naive_acc:.1f}% accuracy')

# baseline 2: predict each pitcher's most common pitch type
sr_pitcher_mode = df_main.groupby('pitcher_id')['pitch_type'].agg(lambda x: x.mode()[0])
df_main['pitcher_mode'] = df_main['pitcher_id'].map(sr_pitcher_mode)
flt_pitcher_acc = (df_main['pitch_type'] == df_main['pitcher_mode']).mean() * 100
print(f'Baseline 2 - Predict each pitcher\'s most common type: {flt_pitcher_acc:.1f}% accuracy')

# baseline 3: predict each pitcher's most common pitch type per count
sr_pitcher_count_mode = df_main.groupby(['pitcher_id', 'count'])['pitch_type'].agg(lambda x: x.mode()[0])
df_main['pitcher_count_mode'] = df_main.set_index(['pitcher_id', 'count']).index.map(sr_pitcher_count_mode)
flt_pitcher_count_acc = (df_main['pitch_type'] == df_main['pitcher_count_mode']).mean() * 100
print(f'Baseline 3 - Predict pitcher\'s most common type per count: {flt_pitcher_count_acc:.1f}% accuracy')

#### Temporal Coverage (for Train/Test Split Planning)

In [ ]:
# date range and games per month
df_main['date'] = pd.to_datetime(df_main['date'])
print(f'Date range: {df_main["date"].min()} to {df_main["date"].max()}')
print(f'Unique games: {df_main["game_pk"].nunique()}')
print(f'Unique pitchers: {df_main["pitcher_id"].nunique()}')
print(f'Unique batters: {df_main["batter_id"].nunique()}')

# pitches per month
df_monthly = df_main.groupby(df_main['date'].dt.to_period('M')).size()
fig, ax = plt.subplots(figsize=(10, 4))
df_monthly.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Pitches per Month')
ax.set_xlabel('Month')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(f'{str_output}/pitches_per_month.png', dpi=150)
plt.show()

#### Key EDA Takeaways

1. **Target class imbalance**: FF (fastball) dominates at ~33%, while rare types (KN, EP, SC) are <1%. Consider grouping rare types or dropping them.
2. **Count matters**: Pitchers throw more fastballs on hitter-friendly counts (3-0, 3-1) and more breaking balls on pitcher-friendly counts (0-2, 1-2).
3. **Handedness matchup**: Pitch mix shifts based on L/R pitcher-batter matchup (e.g., more sliders vs same-hand batters).
4. **Pitcher identity is king**: Each pitcher has a unique repertoire. Pitcher-level features (overall mix, mix per count) will be the strongest predictors.
5. **First pitch bias**: Fastballs are overrepresented on the first pitch of an at-bat.
6. **Fatigue**: Pitch mix changes slightly as pitch count climbs (more fastballs early, mix shifts later).
7. **Baselines to beat**: Naive "always FF" ~33%, pitcher-mode ~45-50%, pitcher-mode-per-count ~50%+.